In [45]:
import sys
sys.path.append("../src/")
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [46]:
from ipm_newer.ipm_solve_no_split import ipm_solve_no_split # type: ignore[reportMissingImports]
from generators.toy_problem_to_json import toy_problem_to_json # type: ignore[reportMissingImports]
from relaxers.quadratic_lin_approx_no_surrogate import quadratic_lin_approx_no_surrogate # type: ignore[reportMissingImports]
from relaxers.quadratic_lin_approx_nosur_refine import quadratic_lin_approx_no_surrogate_refine # type: ignore[reportMissingImports]
from testers.classical_solve_lin_program import solve_lp_return_x # type: ignore[reportMissingImports]
from testers.classical_solve_quad_program import solve_qcp_return_x # type: ignore[reportMissingImports]
from testers.condition_number_lin_program import condition_number_nes_basic # type: ignore[reportMissingImports]

import matplotlib.pyplot as plt
import numpy as np

In [47]:
# Number of copies of the network in the toy system
N=100

# Whether the production and flow variables have upper bounds
F_UPPER_BOUNDS=True

# Whether the demand constraint is an equality or inequality
DEMAND_INEQUALITY=True

# Capacity / demand parameters
# GAMMA=1.0
GAMMA=1.0
LAMBD=1.0
EPS=0.1
DELTA=0.1

# Lower bounds and upper bound on pressure
LX=LP=LF=0.0
UP=0.5

LIN_APPROX = 5

# Whether to do an inner or outer approximation, THIS SHOULD STAY TRUE HERE
OUTER_APPROXIMATION = True

# Whether the coefficient is part of the surrogate
COEFFICIENT_SURROGATE = True

# Whether to bound the surrogates with bound constraints.
SURROGATE_BOUND_BELOW = True
SURROGATE_BOUND_ABOVE = True

# Whether to divide by lambda in an inner approximation
REMOVE_DIVISION = False

In [48]:
toy_problem_to_json(N, F_UPPER_BOUNDS, DEMAND_INEQUALITY, GAMMA, LAMBD,
                        EPS, DELTA, LP, LF, UP)
quad, q_time, q_x = solve_qcp_return_x(f_name="toy.json")
print(f"Exact objective value: {quad}")
print(f"Exact solution: {q_x}")

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26300.2))

CPU model: AMD Ryzen 7 8845HS w/ Radeon 780M Graphics, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Presolve a model with 0 rows, 999 columns and 0 nonzeros (Min)
Model fingerprint: 0x9de527a8
Model has 300 linear objective coefficients
Model has 699 quadratic constraints
Coefficient statistics:
  Matrix range     [0e+00, 0e+00]
  QMatrix range    [1e+00, 1e+00]
  QLMatrix range   [1e+00, 1e+00]
  Objective range  [9e-01, 1e+00]
  Bounds range     [5e-01, 1e+00]
  RHS range        [0e+00, 0e+00]
  QRHS range       [1e+00, 1e+00]
Presolve removed 0 rows and 300 columns
Statistics for model 'qcp_from_json_pre':
  Problem type                : QCP
  Linear constraint matrix    : 1697 rows, 1697 columns, 3392 nonzeros
  Variable types              : 1697 continuous, 0 integer (0 binary)
  Quadratic constraints       : 499 constrs, 1497

In [49]:
quadratic_lin_approx_no_surrogate(LIN_APPROX, OUTER_APPROXIMATION, remove_division=REMOVE_DIVISION,
                                    f_name="toy.json", endpoints=False)
val, time, x, compl = ipm_solve_no_split(f_name="linear_approx.json")
print(x)
print(condition_number_nes_basic())

print(f"Error: {val - quad}")

Solution is below target precision.
The algorithm stopped after 47 iterations in 0.21 seconds.

Primal objective:        -1.10974477e+02
Dual objective:          -1.10974477e+02

Norm of Primal residual: 3.48e-15
Norm of Dual residual:   3.34e-15
Abs. Compl. Gap:         3.22e-09
Rel. Compl. Gap:         1.44e-11

[5.00000000e-01 4.56247043e-01 4.65236925e-01 5.00000000e-01 4.55342958e-01 4.65251260e-01 5.00000000e-01 4.55314814e-01 4.65273618e-01 5.00000000e-01 4.55324117e-01 4.65289147e-01 5.00000000e-01
 4.55329776e-01 4.65299806e-01 5.00000000e-01 4.55332144e-01 4.65307245e-01 5.00000000e-01 4.55332926e-01 4.65312462e-01 5.00000000e-01 4.55333019e-01 4.65316126e-01 5.00000000e-01 4.55332836e-01
 4.65318701e-01 5.00000000e-01 4.55332565e-01 4.65320514e-01 5.00000000e-01 4.55332289e-01 4.65321794e-01 5.00000000e-01 4.55332041e-01 4.65322701e-01 5.00000000e-01 4.55331833e-01 4.65323345e-01
 5.00000000e-01 4.55331658e-01 4.65323805e-01 5.00000000e-01 4.55331523e-01 4.65324134e-01 5.000

In [50]:
# Define a set of functions for each

quads = [[val] for val in x[(5 * N):]]
# print(quads)
iters = 5

for i in range(1, iters + 1):
    print(f"Iteration {i}")
    points_functions = []

    def np_uniform_add_q_factory(qs):
        def np_uniform_add_qs(lower, upper, num):
                refinement_points = len(qs)
                uniform = np.linspace(lower, upper, num - refinement_points)
                return np.append(uniform, qs)
        return np_uniform_add_qs

    for qs in quads:
        points_functions.append(np_uniform_add_q_factory(qs))

    quadratic_lin_approx_no_surrogate_refine(LIN_APPROX + i, OUTER_APPROXIMATION, points_function=points_functions,
                                                remove_division=REMOVE_DIVISION, f_name="toy.json")
    # quadratic_lin_approx_no_surrogate(LIN_APPROX, OUTER_APPROXIMATION,
    #                                             remove_division=REMOVE_DIVISION, f_name="toy.json", endpoints=False)

    val, time, x, compl = ipm_solve_no_split(f_name="linear_approx.json")

    # print(f"x: {x}")
    print(f"val: {val}")
    print(f"Error: {val - quad}")

    for j, qs in enumerate(quads):
        qs.append(x[(5 * N) + j])

    # print(quads)

# print(x)
print(f"Error: {val - quad}")

Iteration 1
Solution is below target precision.
The algorithm stopped after 52 iterations in 0.24 seconds.

Primal objective:        -1.10991072e+02
Dual objective:          -1.10991072e+02

Norm of Primal residual: 3.09e-15
Norm of Dual residual:   3.25e-15
Abs. Compl. Gap:         2.07e-09
Rel. Compl. Gap:         9.27e-12

val: 110.99107165672325
Error: -0.008943089092667833
Iteration 2
The solution quality is limited by the precision of the linear system solver.
The algorithm stopped after 70 iterations in 0.35 seconds.

Primal objective:        -1.10999998e+02
Dual objective:          -1.10999998e+02

Norm of Primal residual: 5.67e-15
Norm of Dual residual:   9.85e-15
Abs. Compl. Gap:         1.91e-08
Rel. Compl. Gap:         8.55e-11

val: 110.99999801108618
Error: -1.6734729740619514e-05
Iteration 3
The solution quality is limited by the precision of the linear system solver.
The algorithm stopped after 98 iterations in 0.53 seconds.

Primal objective:        -1.11000000e+02
Dua